In [ ]:
!gdown 1W-T0iVSPm_6txYUhJqoaC16nnbX88bhu -O data/materias_embeddings.pkl

^C
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/bin/gdown", line 8, in <module>
    sys.exit(main())
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/gdown/__main__.py", line 172, in main
    download(
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/gdown/download.py", line 202, in download
    res = sess.get(url, stream=True, verify=verify)
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/requests/sessions.py", line 602, in get
    return self.request("GET", url, **kwargs)
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/requests/sessions.py", line 589, in request
    resp = self.send(prep, **send_kwargs)
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/requests/sessions.py", line 724, in send
    history = [resp for resp in gen]
  File "/Libra

In [1]:
import numpy as np
import pandas as pd
from scipy import sparse
from lightfm import LightFM
from lightfm.data import Dataset
from sklearn.preprocessing import MultiLabelBinarizer

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


# Data loading

In [2]:
materias_emb = pd.read_pickle("data/materias_embeddings.pkl")

aud = pd.read_csv(
    "data/audiencies.csv",
    usecols=["audiencia_id", "sujeto_pasivo_id", "institucion_id", "materias_tratadas", "fecha"]
)
act = pd.read_csv(
    "data/active_subjects.csv",
    usecols=["audiencia_id", "Nombre completo", "Representa a", "Calidad"]
)
pas = pd.read_csv(
    "data/passive_subjects.csv",
    usecols=["id", "nombre", "cargo", "institution_id"]
).rename(columns={"id": "sujeto_pasivo_id"})

In [3]:
import hashlib

def _norm(s):
    return (str(s) if pd.notna(s) else "").strip().lower()

def make_hash_id(s):
    return int(hashlib.md5(s.encode("utf-8")).hexdigest(), 16) % (10 ** 9)

In [4]:
act["user_id"] = (
    act["Nombre completo"].map(_norm) + "|" +
    act["Representa a"].map(_norm) + "|" +
    act["Calidad"].map(_norm)
)

act["user_id"] = act["user_id"].apply(make_hash_id)

# User and items features

In [5]:
# Users, se crean ids numéricos para los sujetos activos
users = act.loc[:, ["user_id"]].drop_duplicates().copy()
users["u_idx"] = users["user_id"].astype("category").cat.codes

user_id_to_idx = dict(zip(users["user_id"], users["u_idx"]))
user_idx_to_id = dict(enumerate(users["user_id"]))

# Items, se crean ids numéricos para los sujetos pasivos
items = (
    aud.loc[:, ["sujeto_pasivo_id"]].drop_duplicates().rename(columns={"sujeto_pasivo_id": "item_id"})
)
items["i_idx"] = items["item_id"].astype("category").cat.codes

item_id_to_idx = dict(zip(items["item_id"], items["i_idx"]))
item_idx_to_id = dict(enumerate(items["item_id"]))


In [6]:
def split_userwise_holdout(
    act: pd.DataFrame,
    aud: pd.DataFrame,
    test_frac: float = 0.2,
    min_test: int = 1,
    strategy: str = "recent",
    seed: int = 42
):
    """
    Devuelve (train_aud_ids, test_aud_ids) usando un hold-out por usuario.
    - Para cada user_id, separa ~test_frac de sus audiencias (al menos min_test si tiene >=2).
    - strategy="recent": manda las más recientes al test (requiere aud.fecha; si falta, cae a random).
    - strategy="random": selección aleatoria por usuario.
    - Usuarios con 1 audiencia -> van a train (no se puede testear por usuario).
    """
    rng = np.random.default_rng(seed)
    use_recent = (strategy == "recent") and ("fecha" in aud.columns)
    if use_recent:
        aud = aud.copy()
        aud["fecha"] = pd.to_datetime(aud["fecha"], errors="coerce")
    df = act[["audiencia_id", "user_id"]].merge(
        aud[["audiencia_id", "fecha"]] if "fecha" in aud.columns else aud[["audiencia_id"]],
        on="audiencia_id",
        how="left"
    )

    train_ids = []
    test_ids = []

    for uid, g in df.groupby("user_id"):
        ids = g["audiencia_id"].to_numpy()
        n = len(ids)

        if n <= 1:
            train_ids.extend(ids.tolist())
            continue
        k = max(min_test, int(np.floor(test_frac * n)))
        k = min(k, n - 1) 

        if use_recent and g["fecha"].notna().any():
            g2 = g.sort_values("fecha")
            test_pick = g2["audiencia_id"].tail(k).to_numpy()
        else:
            pick_idx = rng.choice(n, size=k, replace=False)
            test_pick = ids[pick_idx]

        train_pick = np.setdiff1d(ids, test_pick, assume_unique=False)

        test_ids.extend(test_pick.tolist())
        train_ids.extend(train_pick.tolist())
        
    train_ids = list(dict.fromkeys(train_ids))
    test_ids  = list(dict.fromkeys(test_ids))
    return train_ids, test_ids


In [7]:
def build_user_embedding_features_from_ids(
    emb_df: pd.DataFrame,
    act: pd.DataFrame,
    aud: pd.DataFrame,
    user_id_to_idx: dict,
    train_aud_ids: list,
    use_recency: bool = True,
    half_life_days: float = 180.0
) -> sparse.csr_matrix:
    tmp = (
        emb_df.loc[emb_df["audiencia_id"].isin(train_aud_ids), ["audiencia_id", "embedding"]]
        .merge(act[["audiencia_id", "user_id"]], on="audiencia_id", how="inner")
        .dropna(subset=["user_id", "embedding"])
    )

    if tmp.empty:
        n_users = (max(user_id_to_idx.values()) + 1) if user_id_to_idx else 0
        emb_dim = len(emb_df["embedding"].iloc[0]) if len(emb_df) else 0
        return sparse.csr_matrix((n_users, emb_dim), dtype="float32")

    if use_recency and ("fecha" in aud.columns):
        tmp = tmp.merge(aud[["audiencia_id", "fecha"]], on="audiencia_id", how="left")
        tmp["fecha"] = pd.to_datetime(tmp["fecha"], errors="coerce")
        tmp = tmp.loc[tmp["fecha"].notna()].copy()

        if tmp.empty:
            tmp["__w__"] = 1.0
        else:
            lam = np.log(2) / float(half_life_days)
            max_date = pd.Timestamp(tmp["fecha"].max())
            f_np = tmp["fecha"].to_numpy(dtype="datetime64[ns]")
            max_np = max_date.to_datetime64()
            ages = (max_np - f_np).astype("timedelta64[D]").astype("float32")
            ages = np.clip(ages, a_min=0, a_max=None)
            tmp["__w__"] = np.exp(-lam * ages)
    else:
        tmp["__w__"] = 1.0

    emb_dim = len(tmp["embedding"].iloc[0])
    E = np.vstack(tmp["embedding"].values).astype("float32")
    W = tmp["__w__"].to_numpy("float32")

    tmp = tmp[["user_id"]].assign(__row__=np.arange(len(tmp)))
    groups = tmp.groupby("user_id")["__row__"].apply(list)

    user_ids = groups.index.to_numpy()
    U_stack = np.zeros((len(user_ids), emb_dim), dtype="float32")
    for j, idxs in enumerate(groups.values):
        ww = W[idxs][:, None]
        vecs = E[idxs, :]
        num = (vecs * ww).sum(axis=0)
        den = ww.sum(axis=0)
        den = np.where(den == 0, 1.0, den)
        U_stack[j, :] = num / den

    # normalización L2
    norms = np.linalg.norm(U_stack, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    U_stack = U_stack / norms

    n_users = (max(user_id_to_idx.values()) + 1) if user_id_to_idx else 0
    U_mat = np.zeros((n_users, emb_dim), dtype="float32")
    u_idx_vec = pd.Series(user_ids).map(user_id_to_idx).dropna().astype(int).to_numpy()
    U_mat[u_idx_vec] = U_stack[:len(u_idx_vec), :]
    return sparse.csr_matrix(U_mat)


In [8]:
train_ids, test_ids = split_userwise_holdout(
    act=act,
    aud=aud,
    test_frac=0.2,     
    min_test=1,        
    strategy="recent",
    seed=42
)

U = build_user_embedding_features_from_ids(
    emb_df=materias_emb,    
    act=act,                
    aud=aud,                
    user_id_to_idx=user_id_to_idx,
    train_aud_ids=train_ids, 
    use_recency=True,
    half_life_days=180
)

/var/folders/pc/1tbslm8954q5q5cyk947n34h0000gn/T/ipykernel_30314/2181257871.py:20: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  aud["fecha"] = pd.to_datetime(aud["fecha"], errors="coerce")
/var/folders/pc/1tbslm8954q5q5cyk947n34h0000gn/T/ipykernel_30314/1856812066.py:23: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  tmp["fecha"] = pd.to_datetime(tmp["fecha"], errors="coerce")


Passive subjects (users) features are the means of the embeddings of their audiencies' materias

In [9]:
def l2_normalize_rows(mat: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return mat / norms


def build_item_embedding_features_from_ids(
    emb_df: pd.DataFrame,      
    aud: pd.DataFrame,         
    item_id_to_idx: dict,      
    train_aud_ids: list,       
    use_recency: bool = True,  
    half_life_days: float = 180.0
) -> sparse.csr_matrix:
    """
    Crea la matriz de features de item (autoridades/pasivos) usando embeddings
    promedio de audiencias en TRAIN, con ponderación por recencia opcional.
    """

    # --- 1) Filtrar solo audiencias del split TRAIN y unir con items ---
    tmp = (
        emb_df.loc[emb_df["audiencia_id"].isin(train_aud_ids), ["audiencia_id", "embedding"]]
        .merge(
            aud[["audiencia_id", "sujeto_pasivo_id", "fecha"]],
            on="audiencia_id", how="inner"
        )
        .rename(columns={"sujeto_pasivo_id": "item_id"})
        .dropna(subset=["item_id", "embedding"])
    )

    # --- 2) Si no hay registros válidos, devolver matriz vacía ---
    if tmp.empty:
        n_items = (max(item_id_to_idx.values()) + 1) if item_id_to_idx else 0
        emb_dim = len(emb_df["embedding"].iloc[0]) if len(emb_df) else 0
        return sparse.csr_matrix((n_items, emb_dim), dtype="float32")

    # --- 3) Calcular pesos por recencia (opcional) ---
    if use_recency and ("fecha" in tmp.columns):
        tmp["fecha"] = pd.to_datetime(tmp["fecha"], errors="coerce")
        tmp = tmp.loc[tmp["fecha"].notna()].copy()

        if tmp.empty:
            tmp["__w__"] = 1.0
        else:
            lam = np.log(2) / float(half_life_days)
            max_date = pd.Timestamp(tmp["fecha"].max())
            f_np = tmp["fecha"].to_numpy(dtype="datetime64[ns]")
            max_np = max_date.to_datetime64()
            ages = (max_np - f_np).astype("timedelta64[D]").astype("float32")
            ages = np.clip(ages, a_min=0, a_max=None)
            tmp["__w__"] = np.exp(-lam * ages)
    else:
        tmp["__w__"] = 1.0

    emb_dim = len(tmp["embedding"].iloc[0])
    E = np.vstack(tmp["embedding"].values).astype("float32")
    W = tmp["__w__"].to_numpy("float32")

    tmp = tmp[["item_id"]].assign(__row__=np.arange(len(tmp)))
    groups = tmp.groupby("item_id")["__row__"].apply(list)

    item_ids = groups.index.to_numpy()
    I_stack = np.zeros((len(item_ids), emb_dim), dtype="float32")

    for j, idxs in enumerate(groups.values):
        ww = W[idxs][:, None]             # (m,1)
        vecs = E[idxs, :]                 # (m,d)
        num = (vecs * ww).sum(axis=0)
        den = ww.sum(axis=0)
        den = np.where(den == 0, 1.0, den)
        I_stack[j, :] = num / den

    I_stack = l2_normalize_rows(I_stack)

    n_items = (max(item_id_to_idx.values()) + 1) if item_id_to_idx else 0
    I_mat = np.zeros((n_items, emb_dim), dtype="float32")
    i_idx_vec = pd.Series(item_ids).map(item_id_to_idx).dropna().astype(int).to_numpy()
    I_mat[i_idx_vec] = I_stack[:len(i_idx_vec), :]

    return sparse.csr_matrix(I_mat)


In [10]:
I = build_item_embedding_features_from_ids(
    emb_df=materias_emb,
    aud=aud,
    item_id_to_idx=item_id_to_idx,
    train_aud_ids=train_ids,
    use_recency=True,
    half_life_days=180
)


/var/folders/pc/1tbslm8954q5q5cyk947n34h0000gn/T/ipykernel_30314/1798735459.py:39: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  tmp["fecha"] = pd.to_datetime(tmp["fecha"], errors="coerce")


# Interactions

In [11]:
from scipy import sparse
import numpy as np
import pandas as pd

def build_interactions_from_ids(
    aud, act, user_id_to_idx, item_id_to_idx, audience_ids, return_df=False
):
    # 1) join básico
    df = (
        aud.loc[aud["audiencia_id"].isin(audience_ids), ["audiencia_id","sujeto_pasivo_id"]]
        .merge(act[["audiencia_id","user_id"]], on="audiencia_id", how="inner")
    )

    # 2) mapear a índices; descartar NaN de mapping
    df["u_idx"] = df["user_id"].map(user_id_to_idx)
    df["i_idx"] = df["sujeto_pasivo_id"].map(item_id_to_idx)
    df = df.dropna(subset=["u_idx","i_idx"]).copy()

    # 3) asegurar enteros nativos (no 'Int64' nullable)
    df["u_idx"] = df["u_idx"].astype(np.int64)
    df["i_idx"] = df["i_idx"].astype(np.int64)

    # 4) eliminar duplicados exactos (u,i) y consolidar (por si hay múltiples audiencias)
    df = df.groupby(["u_idx","i_idx"], as_index=False).size()  # cuenta ocurrencias
    # si quieres binario:
    df["data"] = 1.0
    # si quisieras ponderar por recencia/veces, usa 'size' como peso:
    # df["data"] = df["size"].astype(np.float32)

    # 5) dimensiones de la matriz (basadas en mapeos)
    n_users = int(max(user_id_to_idx.values()) + 1) if user_id_to_idx else 0
    n_items = int(max(item_id_to_idx.values()) + 1) if item_id_to_idx else 0

    # 6) construir CSR
    mat = sparse.csr_matrix(
        (df["data"].to_numpy(np.float32), (df["u_idx"].to_numpy(), df["i_idx"].to_numpy())),
        shape=(n_users, n_items),
        dtype=np.float32
    )
    return (mat, df[["u_idx","i_idx"]].copy()) if return_df else mat


# --- construir y limpiar test respecto de train ---
train, train_df = build_interactions_from_ids(aud, act, user_id_to_idx, item_id_to_idx, train_ids, return_df=True)
test,  test_df  = build_interactions_from_ids(aud, act, user_id_to_idx, item_id_to_idx, test_ids,  return_df=True)

# quitar pares (u,i) que están en TRAIN
u_train = train_df["u_idx"].astype(np.int64).to_numpy()
i_train = train_df["i_idx"].astype(np.int64).to_numpy()
u_test  = test_df["u_idx"].astype(np.int64).to_numpy()
i_test  = test_df["i_idx"].astype(np.int64).to_numpy()

key_train = (u_train << 32) + i_train
key_test  = (u_test  << 32) + i_test

mask = ~np.isin(key_test, key_train)

test_df = test_df.loc[mask]

# rehacer TEST ya limpio
n_users, n_items = train.shape
test = sparse.csr_matrix(
    (np.ones(len(test_df), dtype=np.float32), (test_df["u_idx"].to_numpy(), test_df["i_idx"].to_numpy())),
    shape=(n_users, n_items),
    dtype=np.float32
)


# Trainning

In [12]:
def describe_sparse_matrix(name, mat):
    """Imprime dimensiones y densidad de una matriz CSR."""
    n_rows, n_cols = mat.shape
    nnz = mat.nnz
    density = nnz / (n_rows * n_cols) if n_rows * n_cols > 0 else 0
    print(f"{name}: {n_rows} × {n_cols}  |  nnz={nnz:,}  |  densidad={density:.6f}")

describe_sparse_matrix("Interacciones TRAIN", train)
describe_sparse_matrix("Interacciones TEST", test)
describe_sparse_matrix("User features U", U)
describe_sparse_matrix("Item features I", I)


Interacciones TRAIN: 712057 × 20481  |  nnz=953,972  |  densidad=0.000065
Interacciones TEST: 712057 × 20481  |  nnz=46,612  |  densidad=0.000003
User features U: 712057 × 512  |  nnz=364,525,282  |  densidad=0.999869
Item features I: 20481 × 512  |  nnz=10,221,055  |  densidad=0.974708


In [13]:
import os, time
import numpy as np
import pandas as pd
from lightfm import LightFM
from lightfm.evaluation import precision_at_k, auc_score
from scipy.sparse import csr_matrix

def run_lightfm_experiment(
    train,
    test,
    U,
    I,
    model_hparams,
    n_epochs=10,
    eval_every=2,
    k_eval=10,
    max_train_users=5000,
    min_interactions_per_user_train=5,
    sample_users_max=1000,
    patience=3,
    use_early_stopping=True,
    num_threads=None,
    seed_train=42,
    seed_eval=123,
):
    """
    Entrena un modelo LightFM con muestreo de usuarios para ir rápido
    y devuelve métricas + info de los hiperparámetros probados.
    NO modifica las matrices originales (usa copias internas).
    """

    if num_threads is None:
        num_threads = min(8, (os.cpu_count() or 4))

    train_local = train.tocsr().copy()
    test_local = test.tocsr().copy()
    U_local = csr_matrix(U).copy()   

    user_nnz = np.diff(train_local.indptr)
    eligible_train_users = np.where(user_nnz >= min_interactions_per_user_train)[0]

    if max_train_users is not None and eligible_train_users.size > max_train_users:
        rng_train = np.random.default_rng(seed_train)
        sampled_train_users = np.sort(
            rng_train.choice(eligible_train_users, size=max_train_users, replace=False)
        )
    else:
        sampled_train_users = np.sort(eligible_train_users)

    train_local = train_local[sampled_train_users, :]
    test_local = test_local[sampled_train_users, :]
    U_local = U_local[sampled_train_users, :]

    test_csr = test_local.tocsr()
    eligible_eval_users = np.unique(test_csr.nonzero()[0])

    if eligible_eval_users.size > 0:
        rng_eval = np.random.default_rng(seed_eval)
        sample_size = min(sample_users_max, eligible_eval_users.size)
        user_sample = np.sort(
            rng_eval.choice(eligible_eval_users, size=sample_size, replace=False)
        )
    else:
        user_sample = None

    model = LightFM(**model_hparams)

    best_p10 = -np.inf
    best_auc = -np.inf
    epochs_no_improve = 0

    history = []

    for epoch in range(1, n_epochs + 1):
        t0 = time.time()

        model.fit_partial(
            interactions=train_local,
            user_features=U_local,
            item_features=I,
            epochs=1,
            num_threads=num_threads,
            verbose=False,
        )
        dt = time.time() - t0

        do_eval = (epoch == 1) or (epoch % eval_every == 0) or (epoch == n_epochs)

        if do_eval and user_sample is not None:
            test_sub = test_local[user_sample, :]
            train_sub = train_local[user_sample, :]
            U_sub = U_local[user_sample, :]

            P10 = precision_at_k(
                model,
                test_sub,
                train_interactions=train_sub,
                user_features=U_sub,
                item_features=I,
                k=k_eval,
                num_threads=num_threads,
            ).mean()

            AUC = auc_score(
                model,
                test_sub,
                train_interactions=train_sub,
                user_features=U_sub,
                item_features=I,
                num_threads=num_threads,
            ).mean()
        else:
            P10, AUC = np.nan, np.nan

        history.append(
            {
                "epoch": epoch,
                "time_epoch_sec": dt,
                "P@{}".format(k_eval): float(P10),
                "AUC": float(AUC),
            }
        )

        if use_early_stopping and not np.isnan(P10):
            if P10 > best_p10 + 1e-6:
                best_p10 = P10
                best_auc = AUC
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    break

    result = {
        **model_hparams,
        "n_epochs": len(history),
        "best_P@{}".format(k_eval): float(best_p10),
        "best_AUC": float(best_auc),
        "train_users_used": int(train_local.shape[0]),
        "items_used": int(train_local.shape[1]),
    }

    return result, pd.DataFrame(history)


In [14]:
from itertools import product

losses = ["warp", "bpr"]
no_components_list = [32, 64]
learning_rates = [0.01, 0.05]
user_alphas = [1e-6]
item_alphas = [1e-6]

config_rows = []
results_rows = []

exp_id = 0

for loss, no_components, lr, ua, ia in product(
    losses, no_components_list, learning_rates, user_alphas, item_alphas
):
    exp_id += 1
    print(f"\n===== Experimento {exp_id} =====")
    print(f"loss={loss}, no_components={no_components}, lr={lr}")

    hparams = dict(
        loss=loss,
        no_components=no_components,
        learning_rate=lr,
        user_alpha=ua,
        item_alpha=ia,
        random_state=42,
    )

    result, history = run_lightfm_experiment(
        train=train,
        test=test,
        U=U,
        I=I,
        model_hparams=hparams,
        n_epochs=8,                  
        eval_every=2,
        k_eval=10,
        max_train_users=2000,      
        min_interactions_per_user_train=5,
        sample_users_max=300,
        patience=3,
        use_early_stopping=True,
        num_threads=8,
    )

    result["exp_id"] = exp_id
    config_rows.append(result)

df_results = pd.DataFrame(config_rows)
df_results = df_results.sort_values(by="best_P@10", ascending=False)

df_results



===== Experimento 1 =====
loss=warp, no_components=32, lr=0.01

===== Experimento 2 =====
loss=warp, no_components=32, lr=0.05

===== Experimento 3 =====
loss=warp, no_components=64, lr=0.01

===== Experimento 4 =====
loss=warp, no_components=64, lr=0.05

===== Experimento 5 =====
loss=bpr, no_components=32, lr=0.01

===== Experimento 6 =====
loss=bpr, no_components=32, lr=0.05

===== Experimento 7 =====
loss=bpr, no_components=64, lr=0.01

===== Experimento 8 =====
loss=bpr, no_components=64, lr=0.05


,loss,no_components,learning_rate,user_alpha,item_alpha,random_state,n_epochs,best_P@10,best_AUC,train_users_used,items_used,exp_id
1,warp,32,0.05,0.000001,0.000001,42,8,0.007000,0.869407,2000,20481,2
3,warp,64,0.05,0.000001,0.000001,42,8,0.006667,0.872956,2000,20481,4
2,warp,64,0.01,0.000001,0.000001,42,6,0.002667,0.829472,2000,20481,3
7,bpr,64,0.05,0.000001,0.000001,42,8,0.002667,0.795254,2000,20481,8
0,warp,32,0.01,0.000001,0.000001,42,6,0.002000,0.814124,2000,20481,1
5,bpr,32,0.05,0.000001,0.000001,42,8,0.002000,0.794697,2000,20481,6
6,bpr,64,0.01,0.000001,0.000001,42,8,0.002000,0.672430,2000,20481,7
4,bpr,32,0.01,0.000001,0.000001,42,6,0.001333,0.621019,2000,20481,5


In [ ]:
import os, time
import numpy as np
from lightfm import LightFM
from lightfm.evaluation import precision_at_k, auc_score

best_hparams = dict(
    loss="warp",
    no_components=32,
    learning_rate=0.05,
    user_alpha=1e-6,
    item_alpha=1e-6,
    random_state=42,
)

n_epochs = 30
eval_every = 2
k_eval = 10
use_early_stopping = True
patience = 3           
num_threads = min(12, (os.cpu_count() or 4))

USE_EVAL_SAMPLE = False     
sample_users_max = 5000      

test_csr = test.tocsr()
eligible_eval_users = np.unique(test_csr.nonzero()[0])

if USE_EVAL_SAMPLE and eligible_eval_users.size > sample_users_max:
    rng = np.random.default_rng(123)
    user_sample = np.sort(
        rng.choice(eligible_eval_users, size=sample_users_max, replace=False)
    )
    print(f"[EVAL SAMPLE] Evaluando métricas en {user_sample.size} usuarios de test.")
else:
    user_sample = None
    print("[EVAL] Se usarán todos los usuarios de test para las métricas.")

model = LightFM(**best_hparams)

best_p10 = -np.inf
best_auc = -np.inf
epochs_no_improve = 0

history_full = []

for epoch in range(1, n_epochs + 1):
    t0 = time.time()

    model.fit_partial(
        interactions=train,
        user_features=U,
        item_features=I,
        epochs=1,
        num_threads=num_threads,
        verbose=True,   
    )

    dt = time.time() - t0

    do_eval = (epoch == 1) or (epoch % eval_every == 0) or (epoch == n_epochs)

    if do_eval:
        if user_sample is not None:
            test_sub = test[user_sample, :]
            train_sub = train[user_sample, :]
            U_sub = U[user_sample, :]
        else:
            test_sub = test
            train_sub = train
            U_sub = U

        P10 = precision_at_k(
            model,
            test_sub,
            train_interactions=train_sub,
            user_features=U_sub,
            item_features=I,
            k=k_eval,
            num_threads=num_threads,
        ).mean()

        AUC = auc_score(
            model,
            test_sub,
            train_interactions=train_sub,
            user_features=U_sub,
            item_features=I,
            num_threads=num_threads,
        ).mean()

        history_full.append(
            {
                "epoch": epoch,
                "time_epoch_sec": dt,
                f"P@{k_eval}": float(P10),
                "AUC": float(AUC),
            }
        )

        print(
            f"[Epoch {epoch:02d}/{n_epochs}] {dt:.2f}s | "
            f"P@{k_eval}={P10:.4f} | AUC={AUC:.4f}"
        )

        if use_early_stopping:
            if P10 > best_p10 + 1e-6:
                best_p10 = P10
                best_auc = AUC
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    print(
                        f"Early stopping: sin mejora en {patience} evaluaciones "
                        f"(mejor P@{k_eval}={best_p10:.4f}, AUC={best_auc:.4f})."
                    )
                    break
    else:
        print(
            f"[Epoch {epoch:02d}/{n_epochs}] {dt:.2f}s (sin evaluación externa)"
        )

print("\n=== Resultados finales (mejor época) ===")
print(f"Mejor P@{k_eval}: {best_p10:.4f}")
print(f"Mejor AUC: {best_auc:.4f}")


[EVAL] Se usarán todos los usuarios de test para las métricas.


Epoch: 100%|██████████| 1/1 [00:55<00:00, 55.54s/it]
